In [ ]:
import os, random
import numpy as np
import torch
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

#random.seed(2)
#np.random.seed(2)
#torch.manual_seed(2)


In [ ]:
# ============================================================
# FULL PROGRAM
# - Stable seeds
# - Biases in W1/W2/W3 (prevents harsh ReLU collapse to axes)
# - Standardize inputs (stable learning + geometry)
# - regularizers:
#     * W1 -> disk-like (shrink ONE variance direction)
#     * W2 -> near-orthogonal (reorientation)
#     * W3 -> thin ribbon (shrink ONE variance direction, NOT 1D line)
# - Output improvements:
#     * simplex_spread_loss (prevents Δ2 collapsing to a line)
#     * vertex_pull_loss (encourages Voronoi-like separation in Δ2)
# - Same 2x4 Plotly figure, each panel rotatable
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# -----------------------------
# (A) Reproducibility
# -----------------------------
SEED = 2
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

















# -----------------------------
# (B) Input Space: Torus dataset + labels
# -----------------------------
R_MAJOR = 2.0
R_MINOR = 0.75

# Painted bands around theta: green | yellow1 | purple | yellow2 | green
Y1_START, Y1_END = 0.20*np.pi, 0.55*np.pi
P_START,  P_END  = 0.55*np.pi, 0.70*np.pi
Y2_START, Y2_END = 0.70*np.pi, 1.05*np.pi

# Sampling
N_POINTS = 14000
NOISE = 0.01

rng = np.random.default_rng(SEED)
theta = rng.uniform(0, 2*np.pi, size=N_POINTS)
phi   = rng.uniform(0, 2*np.pi, size=N_POINTS)

x = (R_MAJOR + R_MINOR*np.cos(phi)) * np.cos(theta)
y = (R_MAJOR + R_MINOR*np.cos(phi)) * np.sin(theta)
z = R_MINOR * np.sin(phi)

P0_raw = np.stack([x, y, z], axis=1).astype(np.float32)
P0_raw += rng.normal(scale=NOISE, size=P0_raw.shape).astype(np.float32)



# Labels: 0=green, 1=yellow, 2=purple
t = theta % (2*np.pi)
labels = np.zeros_like(t, dtype=int)
labels[(t >= Y1_START) & (t < Y1_END)] = 1
labels[(t >= Y2_START) & (t < Y2_END)] = 1
labels[(t >= P_START)  & (t < P_END)]  = 2

# Plotly colors
C_GREEN  = "rgb(0,153,140)"
C_YELLOW = "rgb(242,217,51)"
C_PURPLE = "rgb(92,41,140)"
CLASS_NAMES = {0: "Green", 1: "Yellow", 2: "Purple"}
CLASS_COLORS = {0: C_GREEN, 1: C_YELLOW, 2: C_PURPLE}




# -----------------------------
# (B2) Standardize inputs (critical for stable training + geometry)
# -----------------------------
mu = P0_raw.mean(axis=0, keepdims=True)
sd = P0_raw.std(axis=0, keepdims=True) + 1e-8
P0 = ((P0_raw - mu) / sd).astype(np.float32)

# Torch tensors for training
X = torch.tensor(P0, dtype=torch.float32, device=DEVICE)
y_t = torch.tensor(labels, dtype=torch.long, device=DEVICE)














# -----------------------------
# (C) Model: learn W1..W4 automatically (WITH BIASES in W1..W3)
# -----------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # IMPORTANT: biases ON for W1/W2/W3 to avoid harsh ReLU collapse to axes
        self.W1 = nn.Linear(3, 3, bias=True)
        self.W2 = nn.Linear(3, 3, bias=True)
        self.W3 = nn.Linear(3, 3, bias=True)
        self.W4 = nn.Linear(3, 3, bias=True)

        # Good init for stability
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward_trace(self, x):
        z1 = self.W1(x)
        a1 = F.relu(z1)

        z2 = self.W2(a1)
        a2 = F.relu(z2)

        z3 = self.W3(a2)
        a3 = F.relu(z3)

        logits = self.W4(a3)
        return z1, a1, z2, a2, z3, a3, logits

    def forward(self, x):
        return self.forward_trace(x)[-1]





# -----------------------------
# (C2) Loss helpers / regularizers
# -----------------------------
def var_per_dim(P):
    return torch.var(P, dim=0, unbiased=False)

def ortho_penalty(W):
    # encourages W^T W ≈ I (reorientation-like)
    I = torch.eye(W.shape[0], device=W.device, dtype=W.dtype)
    return torch.sum((W.T @ W - I) ** 2)

def collapse_one_dim_loss(P):
    # Rotation-invariant: shrink the smallest variance direction -> ~2D
    v = var_per_dim(P)
    return torch.min(v)

def simplex_spread_loss(logits):
    # Prevent logits from becoming ~1D (which makes Δ2 a line)
    Z = logits - logits.mean(dim=0, keepdim=True)
    C = (Z.T @ Z) / (Z.shape[0] + 1e-8)      # 3x3 covariance
    eig = torch.linalg.eigvalsh(C)           # sorted ascending
    return 1.0 / (eig[-2] + 1e-6)            # penalize tiny 2nd-largest eigenvalue

def vertex_pull_loss(probs, y, n_classes=3):
    # Encourage each class's mean probability vector toward a simplex vertex
    V = torch.eye(n_classes, device=probs.device, dtype=probs.dtype)
    L = 0.0
    for c in range(n_classes):
        pc = probs[y == c]
        if pc.numel() > 0:
            L = L + torch.mean((pc.mean(dim=0) - V[c])**2)
    return L
















# -----------------------------
# (D) Training hyperparameters
# -----------------------------
EPOCHS = 2400
LR = 2e-3

# Paper-story regularizers
LAMBDA_W1_DISK    = 1.0    # W1 -> disk-like (not too strong)
LAMBDA_W2_ORTHO   = 0.05   # W2 -> reorientation
LAMBDA_W3_RIBBON  = 1.2    # W3 -> thin ribbon (keep 2D, NOT a line)

# Δ2 quality regularizers
LAMBDA_SIMPLEX_SPREAD = 0.03   # stop Δ2 degenerating to a line
LAMBDA_VERTEX_PULL    = 0.20   # encourages Voronoi-like class separation

WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

model = Net().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for ep in range(1, EPOCHS + 1):
    opt.zero_grad()

    z1, a1, z2, a2, z3, a3, logits = model.forward_trace(X)

    # (1) Classification loss
    ce = F.cross_entropy(logits, y_t)

    # (2) Paper-style topology regularizers
    disk_loss   = collapse_one_dim_loss(z1)                 # W1 ~ 2D disk
    ribbon_loss = collapse_one_dim_loss(z3)                 # W3 ~ thin 2D ribbon
    ortho_loss  = ortho_penalty(model.W2.weight)            # W2 ~ reorientation

    # (3) Δ2 shape + separation helpers
    spread_loss = simplex_spread_loss(logits)
    probs_train = F.softmax(logits, dim=1)
    vpull_loss  = vertex_pull_loss(probs_train, y_t)

    # (4) Small weight decay
    wd = sum((p*p).sum() for p in model.parameters())

    loss = (ce
            + LAMBDA_W1_DISK   * disk_loss
            + LAMBDA_W3_RIBBON * ribbon_loss
            + LAMBDA_W2_ORTHO  * ortho_loss
            + LAMBDA_SIMPLEX_SPREAD * spread_loss
            + LAMBDA_VERTEX_PULL    * vpull_loss
            + WEIGHT_DECAY     * wd)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if ep % 200 == 0:
        with torch.no_grad():
            acc = (logits.argmax(dim=1) == y_t).float().mean().item()
        print(
            f"epoch {ep:4d} | loss {loss.item():.4f} | acc {acc*100:.2f}% "
            f"| ce {ce.item():.4f} | disk {disk_loss.item():.4f} "
            f"| ribbon {ribbon_loss.item():.4f} | spread {spread_loss.item():.4f} | vpull {vpull_loss.item():.4f}"
        )

print("Training done ✅")




# -----------------------------
# (E) Compute all step outputs (P0..P7)
# -----------------------------
with torch.no_grad():
    z1, a1, z2, a2, z3, a3, logits = model.forward_trace(X)

    # Visualization temperature (ONLY affects appearance of Δ2 plot)
    TEMP = 0.7
    probs = F.softmax(logits / TEMP, dim=1)

P1 = z1.detach().cpu().numpy()
P2 = a1.detach().cpu().numpy()
P3 = z2.detach().cpu().numpy()
P4 = a2.detach().cpu().numpy()
P5 = z3.detach().cpu().numpy()
P6 = a3.detach().cpu().numpy()
P7 = probs.detach().cpu().numpy()

# Use standardized input for Step 1 (better visual consistency)
P0_plot = P0.copy()















# -----------------------------
# (F) Plot all steps in ONE interactive figure (2x4), with legend
# -----------------------------
titles = [
    "Step 1 (Layer 0): Input torus in ℝ³",
    "Step 2 (Layer 1): After learned W1",
    "Step 3 (Layer 2): After ReLU",
    "Step 4 (Layer 3): After learned W2",
    "Step 5 (Layer 4): After ReLU",
    "Step 6 (Layer 5): After learned W3",
    "Step 7 (Layer 6): After ReLU",
    "Step 8 (Final): Softmax output (Δ₂)"
]

points_list = [P0_plot, P1, P2, P3, P4, P5, P6, P7]

fig = make_subplots(
    rows=2, cols=4,
    specs=[[{"type":"scene"}]*4,
           [{"type":"scene"}]*4],
    subplot_titles=titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.08
)

def add_labeled_scatter(scene_row, scene_col, P, show_legend=False):
    for cls in [0, 1, 2]:
        idx = (labels == cls)
        fig.add_trace(
            go.Scatter3d(
                x=P[idx,0], y=P[idx,1], z=P[idx,2],
                mode="markers",
                marker=dict(size=2, color=CLASS_COLORS[cls], opacity=1.0),
                name=CLASS_NAMES[cls],
                showlegend=show_legend
            ),
            row=scene_row, col=scene_col
        )

k = 0
for r in [1, 2]:
    for c in [1, 2, 3, 4]:
        show_leg = (k == 0)
        add_labeled_scatter(r, c, points_list[k], show_legend=show_leg)
        k += 1

# Scene formatting
for i in range(1, 9):
    fig.update_layout(**{
        f"scene{i}": dict(
            xaxis=dict(showgrid=True, zeroline=False, visible=True),
            yaxis=dict(showgrid=True, zeroline=False, visible=True),
            zaxis=dict(showgrid=True, zeroline=False, visible=True),
            aspectmode="data",
            camera=dict(eye=dict(x=1.6, y=1.4, z=0.6))
        )
    })

fig.update_layout(
    title="Neural Network acting on a labeled torus — all layers (interactive, rotatable)",
    width=1400,
    height=800,
    legend=dict(
        x=1.02, y=1.0,
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.2)",
        borderwidth=1
    ),
    margin=dict(l=10, r=220, t=70, b=10)
)

fig.show()


Device: cpu
epoch  200 | loss 1.0566 | acc 65.97% | ce 0.7507 | disk 0.0333 | ribbon 0.0242 | spread 2.9121 | vpull 0.6181
epoch  400 | loss 0.4607 | acc 88.54% | ce 0.3399 | disk 0.0017 | ribbon 0.0005 | spread 1.1295 | vpull 0.3508
epoch  600 | loss 0.3579 | acc 90.60% | ce 0.2690 | disk 0.0040 | ribbon 0.0000 | spread 0.6434 | vpull 0.2849
epoch  800 | loss 0.3045 | acc 90.64% | ce 0.2299 | disk 0.0061 | ribbon 0.0001 | spread 0.4570 | vpull 0.2392
epoch 1000 | loss 0.2662 | acc 95.93% | ce 0.2013 | disk 0.0093 | ribbon 0.0001 | spread 0.3405 | vpull 0.1966
epoch 1200 | loss 0.2357 | acc 97.69% | ce 0.1771 | disk 0.0150 | ribbon 0.0001 | spread 0.2484 | vpull 0.1550
epoch 1400 | loss 0.2100 | acc 97.76% | ce 0.1563 | disk 0.0182 | ribbon 0.0001 | spread 0.2040 | vpull 0.1229
epoch 1600 | loss 0.1879 | acc 97.74% | ce 0.1379 | disk 0.0210 | ribbon 0.0000 | spread 0.1653 | vpull 0.0972
epoch 1800 | loss 0.1680 | acc 97.94% | ce 0.1212 | disk 0.0227 | ribbon 0.0000 | spread 0.1314 | vp